# IC powertrain and driveline

The CT-17 IC's restricted Yamaha R6 (`lapsim/subsystems/power/ic.py`: restrictor flow, volumetric efficiency,
friction, cooling derate) and its driveline (`lapsim/subsystems/driveline.py`: primary gear, 6-speed box,
520 chain, Drexler limited-slip differential, rotating inertia per gear). The traction diagram is the wheel
force in every gear against the resistance; the solver shifts at the speeds where the next gear gives more.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
CAR = lapsim.load_car("ct17_ic")
SL = [slider(CAR, "pt:final_drive.rear_sprocket", 30, 50, 1, "rear sprocket", 1, "teeth"),
      slider(CAR, "pt:final_drive.front_sprocket", 10, 16, 1, "front sprocket", 1, "teeth"),
      slider(CAR, "pt:gearbox.upshift_time", 0.02, 0.20, 0.005, "upshift time", 1, "s"),
      slider(CAR, "pt:engine.torque_scale", 0.85, 1.10, 0.005, "engine torque scale"),
      slider(CAR, "pt:differential.preload", 0, 120, 1, "diff preload", 1, "N m")]
out, out_lap = W.Output(), W.Output()


def draw(*_):
    with out:
        out.clear_output(wait=True)
        veh = with_values(CAR, SL).build()
        rpm = np.arange(4000, veh.ptc.engine.rev_limit + 1, 100.0)
        w = veh.engine.wot(rpm)
        T = lapsim.solvers.qss_ic.ICTables(veh)
        fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
        ax[0].plot(rpm, w["torque"], label="torque [N m]"); ax[0].plot(rpm, w["power"] / 1e3, label="power [kW]")
        ax[0].set(xlabel="rpm", title="Engine at wide-open throttle"); ax[0].legend()
        for k, F in enumerate(T.F, start=1):
            ax[1].plot(3.6 * T.v, F, label=f"gear {k}")
        ax[1].plot(3.6 * T.v, T.R, "k--", label="resistance")
        ax[1].set(xlabel="speed [km/h]", ylabel="N", ylim=(0, None), title="Wheel force by gear"); ax[1].legend(fontsize=8)
        plt.show()
        display(pd.DataFrame(veh.dt.summary(veh.m)).round(3))


def laps():
    display(with_values(CAR, SL).run(("acceleration", "autocross")).table().round(3))


for s in SL:
    s.observe(draw, "value")
draw()
display(W.VBox([*SL, out, run_button("Run acceleration + autocross", laps, out_lap), out_lap]))